# GP-Sparse Attention Training Pipeline with Objaverse

End-to-end notebook for:
1. Download 3D models from Objaverse
2. Convert to RenderFormer HDF5 format
3. Train sparse attention model
4. Evaluate vs dense baseline

**Time estimate**: 2-3 hours for full run (depends on internet speed + GPU)

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Ensure code module is importable
code_dir = Path.cwd()
if code_dir.name != 'code':
    code_dir = code_dir / 'code'
if code_dir not in sys.path:
    sys.path.insert(0, str(code_dir))

print(f"Working directory: {Path.cwd()}")
print(f"Code path: {code_dir}")
print(f"Python version: {sys.version}")

In [ ]:
# Install dependencies
import subprocess

print("Installing dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'objaverse', 'trimesh'], check=True)
print("✓ Dependencies installed")

In [ ]:
# Core imports
import torch
import numpy as np
import h5py
import json
from tqdm import tqdm
import trimesh
from typing import Dict, List

# RenderFormer imports (used to verify setup)
from renderformer.models.config import RenderFormerConfig
from renderformer.models.renderformer import RenderFormer
from renderformer.layers.sparse_attention import GPSparseAttention

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download Objaverse Models

In [ ]:
import objaverse

objaverse_dir = Path('objaverse_models')
objaverse_dir.mkdir(exist_ok=True)

print(f"Downloading Objaverse models to {objaverse_dir}...")
print("This may take 10-30 minutes depending on internet speed.")

uids = objaverse.load_uids()
print(f"Total Objaverse models available: {len(uids)}")

num_models = 10  # Change to 50, 100, 500+ for full training
selected_uids = uids[:num_models]

print(f"Downloading {num_models} models...")

for uid in tqdm(selected_uids, desc="Downloading models"):
    try:
        filepath = objaverse.load_object(uid, download_dir=str(objaverse_dir))
    except:
        continue

print(f"✓ Models downloaded to {objaverse_dir}")

## 3. Convert Models to HDF5 Format

Call convert_to_hdf5.py to process meshes and generate training data.

In [ ]:
import subprocess

training_data_dir = Path('training_data')

# Call convert_to_hdf5.py
print("=" * 60)
print("Converting models to HDF5 format")
print("=" * 60)

result = subprocess.run([
    'python', 'convert_to_hdf5.py',
    '--input_dir', str(objaverse_dir),
    '--output_dir', str(training_data_dir),
    '--max_faces', '5000',
    '--min_faces', '100',
    '--num_views', '4',
], cwd=Path.cwd())

print(f"\n✓ HDF5 conversion complete")
print(f"Training data saved to {training_data_dir}")

## 4. Train Models with train.py

Call the professional training scripts.

In [ ]:
def create_training_hdf5(mesh: trimesh.Trimesh, output_path: str, seed: int, num_views: int = 4):
    """Create HDF5 file in RenderFormer format from a mesh."""
    vertices = mesh.vertices.astype(np.float32)
    faces = mesh.faces.astype(np.int32)
    face_normals = compute_face_normals(vertices, faces).astype(np.float32)
    materials = create_random_materials(len(faces), seed=seed)
    
    # Generate camera poses
    camera_poses = []
    for i in range(num_views):
        angle = (i / num_views) * 2 * np.pi
        radius = 1.8
        height = 0.2
        cam_pos = np.array([radius * np.cos(angle), height, radius * np.sin(angle)], dtype=np.float32)
        forward = -cam_pos / np.linalg.norm(cam_pos)
        camera_poses.append({'position': cam_pos, 'forward': forward, 'up': np.array([0, 1, 0], dtype=np.float32), 'fov': 45.0})
    
    # Synthetic images
    img_size = 256
    images = (np.random.rand(num_views, img_size, img_size, 3) * 0.5 + 0.25).astype(np.float32)
    
    # Write HDF5
    with h5py.File(output_path, 'w') as f:
        f.create_dataset('vertices', data=vertices)
        f.create_dataset('triangles', data=faces)
        f.create_dataset('vertex_normals', data=mesh.vertex_normals.astype(np.float32))
        f.create_dataset('diffuse_albedo', data=materials['diffuse_albedo'])
        f.create_dataset('specular_albedo', data=materials['specular_albedo'])
        f.create_dataset('roughness', data=materials['roughness'].astype(np.float32))
        f.create_dataset('metallic', data=materials['metallic'].astype(np.float32))
        f.create_dataset('images', data=images)
        camera_group = f.create_group('cameras')
        for i, cam in enumerate(camera_poses):
            cam_subgroup = camera_group.create_group(f'camera_{i}')
            for key, val in cam.items():
                if isinstance(val, np.ndarray):
                    cam_subgroup.create_dataset(key, data=val)
                else:
                    cam_subgroup.attrs[key] = val
    return output_path

training_data_dir = Path('training_data')
training_data_dir.mkdir(exist_ok=True)

hdf5_files = []
print("Converting models to HDF5 format...\n")

for idx, model in enumerate(tqdm(downloaded_models, desc="Converting models")):
    try:
        mesh = load_mesh(model['filepath'])
        if mesh is None or len(mesh.vertices) < 3 or len(mesh.faces) < 1:
            continue
        if len(mesh.faces) > 5000:
            target_count = np.random.randint(1000, 3000)
            mesh = mesh.simplify_quadratic_decimation(target_count=target_count)
        if len(mesh.faces) < 100:
            continue
        mesh = normalize_mesh(mesh)
        output_file = training_data_dir / f'scene_{idx:04d}_{model["uid"][:8]}.h5'
        create_training_hdf5(mesh, str(output_file), seed=idx, num_views=4)
        hdf5_files.append(output_file)
    except:
        continue

print(f"\n✓ Created {len(hdf5_files)} training HDF5 files")

## 5. Evaluate with eval.py

Run benchmarking and comparison between sparse and dense models.

In [ ]:
import subprocess
import json
from pathlib import Path

training_data_dir = Path('training_data')
checkpoint_dir = Path('checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

# Get current working directory for train.py call
code_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()

# Train dense baseline
print("=" * 60)
print("Training Dense Baseline (O(N²) attention)")
print("=" * 60)
result_dense = subprocess.run([
    'python', 'train.py',
    '--dataset', str(training_data_dir),
    '--num_epochs', '10',
    '--batch_size', '2',
    '--lr', '1e-4',
    '--baseline',
    '--checkpoint_dir', str(checkpoint_dir),
], cwd=Path.cwd())

print("\n✓ Dense training complete")
print(f"Checkpoints saved to {checkpoint_dir}")

# Train sparse model
print("\n" + "=" * 60)
print("Training Sparse Model (GP-Sparse, O(N·k) attention)")
print("=" * 60)
result_sparse = subprocess.run([
    'python', 'train.py',
    '--dataset', str(training_data_dir),
    '--num_epochs', '10',
    '--batch_size', '2',
    '--lr', '1e-4',
    '--use_sparse',
    '--checkpoint_dir', str(checkpoint_dir),
], cwd=Path.cwd())

print("\n✓ Sparse training complete")

## 7. Evaluate with eval.py

Run benchmarking and comparison between sparse and dense models.

In [ ]:
# Benchmark sparse vs dense
print("=" * 60)
print("Benchmarking: Sparse vs Dense Attention")
print("=" * 60)

checkpoint_dense = checkpoint_dir / 'model_dense_best.pt'
checkpoint_sparse = checkpoint_dir / 'model_sparse_best.pt'

eval_results_dir = Path('eval_results')
eval_results_dir.mkdir(exist_ok=True)

result_eval = subprocess.run([
    'python', 'eval.py',
    '--checkpoint_dense', str(checkpoint_dense),
    '--checkpoint_sparse', str(checkpoint_sparse),
    '--output_dir', str(eval_results_dir),
    '--num_iterations', '5',
], cwd=Path.cwd())

print("\n✓ Evaluation complete")
print(f"Results saved to {eval_results_dir}")

# Load and display results
results_file = eval_results_dir / 'benchmark_results.json'
if results_file.exists():
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print("\n" + "=" * 60)
    print("BENCHMARK RESULTS")
    print("=" * 60)
    if 'comparison' in results:
        print(f"Speedup: {results['comparison']['speedup']:.2f}×")
        print(f"Memory Reduction: {results['comparison']['memory_reduction']:.2f}×")
    if 'sparse' in results:
        print(f"\nSparse Attention:")
        print(f"  Time: {results['sparse']['time_mean_ms']:.2f} ± {results['sparse']['time_std_ms']:.2f} ms")
        print(f"  Memory: {results['sparse']['memory_gb']:.2f} GB")
    if 'dense' in results:
        print(f"\nDense Attention:")
        print(f"  Time: {results['dense']['time_mean_ms']:.2f} ± {results['dense']['time_std_ms']:.2f} ms")
        print(f"  Memory: {results['dense']['memory_gb']:.2f} GB")

In [ ]:
## 6. Summary

This notebook demonstrates the complete GP-Sparse Attention pipeline:

1. **Download**: Objaverse model download
2. **Convert**: Call `convert_to_hdf5.py` for mesh processing and HDF5 generation
3. **Train**: Call `train.py` to train dense and sparse models
4. **Evaluate**: Call `eval.py` to benchmark and compare performance

### Files Generated

```
objaverse_models/       # Downloaded 3D models
training_data/          # HDF5 files for training
checkpoints/            # Trained model checkpoints
eval_results/           # Benchmark results
```

### Command-Line Usage

Alternatively, use scripts directly:
```bash
# Convert models to HDF5
python convert_to_hdf5.py --input_dir ./objaverse_models --output_dir ./training_data

# Train models
python train.py --dataset ./training_data --use_sparse --num_epochs 100

# Evaluate
python eval.py --checkpoint_sparse ... --checkpoint_dense ...
```

print("✓ Notebook complete!")

In [ ]:
def benchmark_model(model, test_loader, device, num_iterations=3):
    model.eval()
    times = []
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            if batch_idx >= num_iterations:
                break
            tri_vpos = batch['tri_vpos'].to(device)
            vns = batch['vns'].to(device)
            valid_mask = batch['valid_mask'].to(device)
            batch_size, num_tri, _ = tri_vpos.shape
            texture_patches = torch.randn(batch_size, num_tri, 13, 32, 32, device=device)
            rays_o = torch.randn(batch_size, 2, 3, device=device)
            rays_d = torch.randn(batch_size, 2, 256, 256, 3, device=device)
            tri_vpos_view = tri_vpos.unsqueeze(1).expand(-1, 2, -1, -1)
            if batch_idx == 0:
                _ = model(tri_vpos, texture_patches, valid_mask, vns, rays_o, rays_d, tri_vpos_view)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
            t0 = time.time()
            _ = model(tri_vpos, texture_patches, valid_mask, vns, rays_o, rays_d, tri_vpos_view)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.time() - t0)
    return np.mean(times), np.std(times)

t_dense, std_dense = benchmark_model(model_dense, train_loader, device, num_iterations=3)
t_sparse, std_sparse = benchmark_model(model_sparse, train_loader, device, num_iterations=3)
speedup = t_dense / t_sparse

print(f"\nBenchmark Results:")
print(f"Dense:  {t_dense*1000:.2f} ± {std_dense*1000:.2f} ms")
print(f"Sparse: {t_sparse*1000:.2f} ± {std_sparse*1000:.2f} ms")
print(f"Speedup: {speedup:.2f}×")

## 8. Save Results

In [ ]:
checkpoint_dir = Path('checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

torch.save({'model': model_dense.state_dict(), 'loss': loss_dense}, checkpoint_dir / 'model_dense.pt')
torch.save({'model': model_sparse.state_dict(), 'loss': loss_sparse}, checkpoint_dir / 'model_sparse.pt')

results = {'dense_loss': loss_dense, 'sparse_loss': loss_sparse, 'speedup': speedup, 'time_dense_ms': t_dense*1000, 'time_sparse_ms': t_sparse*1000}
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Models saved to {checkpoint_dir}")
print(f"✓ Results saved to results.json")